In [67]:
# =============================================================================
# Phase 1 — Spark ETL Pipeline : Bronze → Silver Layer
# Craigslist Used Vehicle Dataset — Data Cleaning
# =============================================================================

In [68]:
import logging
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql import SparkSession, Window   # ← add Window here
from pyspark.sql.functions import upper, broadcast

In [69]:
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.debug.maxToStringFields", "1000")

In [70]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleETL_Silver")

In [71]:
# =============================================================================
# CONFIG
# =============================================================================

In [72]:
BRONZE_PATH         = "hdfs:///user/vehicle_market/bronze/vehicles.csv"
BRONZE_VALIDATED    = "hdfs:///user/vehicle_market/bronze_validated/vehicles_validated"
SILVER_PATH         = "hdfs:///user/vehicle_market/silver/vehicles_clean"

In [73]:
CURRENT_YEAR = datetime.now().year + 1

CATEGORY_B_COLS = ["size", "condition", "cylinders", "drive", "paint_color", "type"]

MANUFACTURER_LOOKUP = {
    # Ford
    "f-150": "ford", "f150": "ford", "mustang": "ford", "explorer": "ford",
    "escape": "ford", "focus": "ford", "fusion": "ford", "ranger": "ford",
    "expedition": "ford", "f-250": "ford", "f-350": "ford", "edge": "ford",
    # Chevrolet
    "silverado": "chevrolet", "camaro": "chevrolet", "malibu": "chevrolet",
    "tahoe": "chevrolet", "suburban": "chevrolet", "equinox": "chevrolet",
    "colorado": "chevrolet", "cruze": "chevrolet", "impala": "chevrolet",
    "blazer": "chevrolet", "traverse": "chevrolet",
    # Toyota
    "camry": "toyota", "corolla": "toyota", "rav4": "toyota",
    "tacoma": "toyota", "highlander": "toyota", "4runner": "toyota",
    "tundra": "toyota", "prius": "toyota", "sienna": "toyota",
    # Honda
    "civic": "honda", "accord": "honda", "cr-v": "honda",
    "pilot": "honda", "odyssey": "honda", "fit": "honda", "hrv": "honda",
    # Nissan
    "altima": "nissan", "sentra": "nissan", "maxima": "nissan",
    "rogue": "nissan", "frontier": "nissan", "pathfinder": "nissan", "murano": "nissan",
    # Dodge
    "ram": "dodge", "charger": "dodge", "challenger": "dodge",
    "durango": "dodge", "dart": "dodge", "caravan": "dodge",
    # Jeep
    "wrangler": "jeep", "cherokee": "jeep", "grand cherokee": "jeep",
    "compass": "jeep", "renegade": "jeep",
    # BMW
    "3 series": "bmw", "5 series": "bmw", "x5": "bmw", "x3": "bmw", "x1": "bmw",
    # Mercedes
    "c-class": "mercedes-benz", "e-class": "mercedes-benz", "glc": "mercedes-benz",
    "gle": "mercedes-benz",
    # Hyundai
    "elantra": "hyundai", "sonata": "hyundai", "tucson": "hyundai",
    "santa fe": "hyundai", "accent": "hyundai",
    # GMC
    "sierra": "gmc", "yukon": "gmc", "terrain": "gmc", "acadia": "gmc",
    # Subaru
    "outback": "subaru", "forester": "subaru", "impreza": "subaru",
    "crosstrek": "subaru", "legacy": "subaru",
    # Kia
    "sorento": "kia", "sportage": "kia", "optima": "kia", "soul": "kia",
    # Volkswagen
    "jetta": "volkswagen", "passat": "volkswagen", "tiguan": "volkswagen",
    "golf": "volkswagen", "atlas": "volkswagen",
}

CATEGORICAL_NORMALIZE_COLS = [
    "fuel", "condition", "drive", "transmission", "manufacturer",
    "title_status", "paint_color", "type", "size", "cylinders"
]

state_lookup = [
    ("AL", "Alabama"),
    ("AK", "Alaska"),
    ("AZ", "Arizona"),
    ("AR", "Arkansas"),
    ("CA", "California"),
    ("CO", "Colorado"),
    ("CT", "Connecticut"),
    ("DE", "Delaware"),
    ("DC", "District of Columbia"),
    ("FL", "Florida"),
    ("GA", "Georgia"),
    ("HI", "Hawaii"),
    ("ID", "Idaho"),
    ("IL", "Illinois"),
    ("IN", "Indiana"),
    ("IA", "Iowa"),
    ("KS", "Kansas"),
    ("KY", "Kentucky"),
    ("LA", "Louisiana"),
    ("ME", "Maine"),
    ("MD", "Maryland"),
    ("MA", "Massachusetts"),
    ("MI", "Michigan"),
    ("MN", "Minnesota"),
    ("MS", "Mississippi"),
    ("MO", "Missouri"),
    ("MT", "Montana"),
    ("NE", "Nebraska"),
    ("NV", "Nevada"),
    ("NH", "New Hampshire"),
    ("NJ", "New Jersey"),
    ("NM", "New Mexico"),
    ("NY", "New York"),
    ("NC", "North Carolina"),
    ("ND", "North Dakota"),
    ("OH", "Ohio"),
    ("OK", "Oklahoma"),
    ("OR", "Oregon"),
    ("PA", "Pennsylvania"),
    ("RI", "Rhode Island"),
    ("SC", "South Carolina"),
    ("SD", "South Dakota"),
    ("TN", "Tennessee"),
    ("TX", "Texas"),
    ("UT", "Utah"),
    ("VT", "Vermont"),
    ("VA", "Virginia"),
    ("WA", "Washington"),
    ("WV", "West Virginia"),
    ("WI", "Wisconsin"),
    ("WY", "Wyoming")
]


In [74]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================


In [75]:
spark = SparkSession.builder \
    .appName("VehicleMarket_ETL_Silver") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

pipeline_start = time.time()
log.info("="*65)
log.info("  Vehicle Market ETL Pipeline — Bronze → Silver")
log.info("="*65)

2026-08-01 03:58:38 | INFO | =================================================================
2026-08-01 03:58:38 | INFO |   Vehicle Market ETL Pipeline — Bronze → Silver
2026-08-01 03:58:38 | INFO | =================================================================


In [76]:
# =============================================================================
# 2. LOAD RAW DATA (Bronze Layer)
# =============================================================================


In [77]:
log.info("Loading raw data from HDFS Bronze layer...")
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("multiLine", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("mode", "PERMISSIVE") \
    .csv(BRONZE_PATH)
record_count_bronze = df_raw.count()
col_count_bronze = len(df_raw.columns)
log.info(f"Bronze layer loaded | Records: {record_count_bronze} | Columns: {col_count_bronze}")


2026-08-01 03:58:39 | INFO | Loading raw data from HDFS Bronze layer...
2026-08-01 03:58:45 | INFO | Bronze layer loaded | Records: 426880 | Columns: 26


In [78]:
# =============================================================================
# 3. PRE-CLEANING DATA PROFILING REPORT
# =============================================================================

In [79]:
log.info("Generating pre-cleaning data profile...")

pre_null_counts = {
    c: df_raw.filter(F.col(c).isNull() | (F.trim(F.col(c)) == "")).count()
    for c in df_raw.columns
}

pre_profile = {
    "distinct_manufacturers" : df_raw.select("manufacturer").distinct().count(),
    "distinct_models"        : df_raw.select("model").distinct().count(),
    "top_fuel_types"         : df_raw.groupBy("fuel").count().orderBy(F.desc("count")).limit(5).collect(),
    "top_states"             : df_raw.groupBy("state").count().orderBy(F.desc("count")).limit(5).collect(),
}

log.info(f"  Distinct Manufacturers : {pre_profile['distinct_manufacturers']}")
log.info(f"  Distinct Models        : {pre_profile['distinct_models']}")
log.info("  Top Fuel Types         : " + str([(r['fuel'], r['count']) for r in pre_profile['top_fuel_types']]))
log.info("  Top States             : " + str([(r['state'], r['count']) for r in pre_profile['top_states']]))


2026-08-01 03:58:45 | INFO | Generating pre-cleaning data profile...
2026-08-01 04:00:43 | INFO |   Distinct Manufacturers : 43                      
2026-08-01 04:00:43 | INFO |   Distinct Models        : 29669
2026-08-01 04:00:43 | INFO |   Top Fuel Types         : [('gas', 356209), ('other', 30728), ('diesel', 30062), ('hybrid', 5170), (None, 3013)]
2026-08-01 04:00:43 | INFO |   Top States             : [('ca', 50614), ('fl', 28511), ('tx', 22945), ('ny', 19386), ('oh', 17696)]


In [80]:
# =============================================================================
# 4. CATEGORY A — DROP HIGH NULL / LOW VALUE COLUMNS
# =============================================================================

In [81]:
cols_dropped = ["county"]
df = df_raw.drop(*cols_dropped)
log.info(f"[Category A] Dropped columns: {cols_dropped}")

2026-08-01 04:00:43 | INFO | [Category A] Dropped columns: ['county']


In [82]:
# =============================================================================
# 5. CATEGORICAL NORMALIZATION (Lowercase + Trim)
# =============================================================================

In [83]:
log.info("Normalizing categorical columns...")
for col in CATEGORICAL_NORMALIZE_COLS:
    if col in df.columns:
        df = df.withColumn(col, F.lower(F.trim(F.col(col))))

# Standardize known variants
transmission_map = {"auto": "automatic", "manual": "manual", "other": "other"}
df = df.withColumn(
    "transmission",
    F.when(F.col("transmission").isin(list(transmission_map.keys())),
           F.col("transmission")).otherwise(F.col("transmission"))
)

log.info(f"  Normalized: {CATEGORICAL_NORMALIZE_COLS}")

2026-08-01 04:00:43 | INFO | Normalizing categorical columns...
2026-08-01 04:00:43 | INFO |   Normalized: ['fuel', 'condition', 'drive', 'transmission', 'manufacturer', 'title_status', 'paint_color', 'type', 'size', 'cylinders']


In [84]:
# =============================================================================
# 6. CATEGORY B — OPTIONAL ATTRIBUTES → "unknown"
# =============================================================================

In [85]:
for col in CATEGORY_B_COLS:
    df = df.withColumn(
        col,
        F.when(F.col(col).isNull() | (F.trim(F.col(col)) == ""), "unknown")
         .otherwise(F.col(col))
    )
log.info(f"[Category B] Filled 'unknown' for: {CATEGORY_B_COLS}")

2026-08-01 04:00:43 | INFO | [Category B] Filled 'unknown' for: ['size', 'condition', 'cylinders', 'drive', 'paint_color', 'type']


In [86]:
# =============================================================================
# 7. VIN — IDENTIFIER FIELD (preserve existing, no imputation)
# =============================================================================
# VIN nulls are intentionally retained. Used only for dedup validation.

In [87]:
log.info("[VIN] Retained as identifier. No imputation applied.")

2026-08-01 04:00:43 | INFO | [VIN] Retained as identifier. No imputation applied.


In [88]:
# =============================================================================
# 8. CATEGORY C — BUSINESS CRITICAL ATTRIBUTES
# =============================================================================

In [89]:
# --- 8a. Manufacturer Inference via contains() matching ---
log.info("Inferring missing manufacturers from model names...")

df = df.withColumn("model_lower", F.lower(F.trim(F.col("model"))))

inferred_manufacturer = F.lit(None).cast("string")
for model_keyword, mfr in MANUFACTURER_LOOKUP.items():
    inferred_manufacturer = F.when(
        F.col("model_lower").contains(model_keyword),
        F.lit(mfr)
    ).otherwise(inferred_manufacturer)

df = df.withColumn(
    "manufacturer",
    F.when(
        F.col("manufacturer").isNull() | (F.col("manufacturer") == ""),
        inferred_manufacturer
    ).otherwise(F.col("manufacturer"))
).withColumn(
    "manufacturer",
    F.when(F.col("manufacturer").isNull(), "unknown").otherwise(F.col("manufacturer"))
).drop("model_lower")

# --- 8b. Model ---
df = df.withColumn(
    "model",
    F.when(F.col("model").isNull() | (F.trim(F.col("model")) == ""), "unknown model")
     .otherwise(F.lower(F.trim(F.col("model"))))
)

# --- 8c. Year — Drop null records ---
rows_before_year_drop = df.count()
df = df.filter(F.col("year").isNotNull() & (F.trim(F.col("year")) != ""))
rows_after_year_drop = df.count()
log.info(f"[Year] Null records removed: {rows_before_year_drop - rows_after_year_drop}")

# --- 8d. Fuel, Title Status, State → "unknown" ---
for col in ["fuel", "title_status", "state"]:   # ← remove "transmission" from here
    df = df.withColumn(
        col,
        F.when(F.col(col).isNull() | (F.trim(F.col(col)) == ""), "unknown")
         .otherwise(F.col(col))
    )

log.info("[Category C] Business critical columns handled.")


2026-08-01 04:00:43 | INFO | Inferring missing manufacturers from model names...
2026-08-01 04:00:51 | INFO | [Year] Null records removed: 1205                  
2026-08-01 04:00:51 | INFO | [Category C] Business critical columns handled.


In [90]:
state_lookup_df = spark.createDataFrame(
    state_lookup,
    ["state_code", "state_name"]
)


df = df.withColumn("state", F.upper(F.col("state")))
df = df.join(
    F.broadcast(state_lookup_df),
    F.col("state") == F.col("state_code"),
    "left"
)
df = df.drop("state_code")

In [91]:
print(df.columns)

['id', 'url', 'region', 'region_url', 'price', 'year', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN', 'drive', 'size', 'type', 'paint_color', 'image_url', 'description', 'state', 'lat', 'long', 'posting_date', 'state_name']


In [92]:
print(df.select(F.col('state_name')).distinct().count())
df.select(F.col('state_name')).distinct().show(51)

51


[Stage 997:>                                                        (0 + 1) / 1]

+--------------------+
|          state_name|
+--------------------+
|            Maryland|
|              Nevada|
|              Hawaii|
|             Arizona|
|             Florida|
|            Michigan|
|      North Carolina|
|                Ohio|
|        Rhode Island|
|         Mississippi|
|            Nebraska|
|            New York|
|      South Carolina|
|             Vermont|
|       West Virginia|
|           Wisconsin|
|              Alaska|
|               Maine|
|        Pennsylvania|
|          Washington|
|            Kentucky|
|        South Dakota|
|            Arkansas|
|          California|
|         Connecticut|
|             Georgia|
|               Idaho|
|            Illinois|
|           Minnesota|
|            Oklahoma|
|                Utah|
|             Alabama|
|       Massachusetts|
|             Montana|
|             Indiana|
|           Louisiana|
|          New Mexico|
|        North Dakota|
|            Virginia|
|            Missouri|
|          

In [93]:
# Break query plan — accumulated transformations from Category A/B/C
spark.sparkContext.setCheckpointDir("hdfs:///user/vehicle_market/logs/checkpoints")
df = df.checkpoint()
log.info("Checkpoint 1 written — query plan reset after Category C.")

2026-08-01 04:01:12 | INFO | Checkpoint 1 written — query plan reset after Category C.


In [94]:
# =============================================================================
# 9. DATA VALIDATION — CLEAN FIELDS BEFORE TYPE CASTING
# =============================================================================

In [95]:
# log.info("Running data validation before type casting...")

# # --- Price Cleaning ---
# # Handles: $5,000 | 5k | 10K | "5000 obo" | "call" | "trade" | "best offer"
# df = df.withColumn("price_str", F.lower(F.trim(F.col("price"))))
# df = df.withColumn("price_str", F.regexp_replace(F.col("price_str"), r"[\$,\s]", ""))
# df = df.withColumn(
#     "price_str",
#     F.when(F.col("price_str").rlike(r"^\d+k$"),
#            (F.regexp_replace(F.col("price_str"), "k", "").cast("int") * 1000).cast("string"))
#     .otherwise(F.col("price_str"))
# )
# # Strip non-numeric suffixes like "obo", "firm"
# df = df.withColumn("price_str", F.regexp_extract(F.col("price_str"), r"^(\d+)", 1))
# df = df.withColumn(
#     "price_clean",
#     F.when(
#     F.col("price_str").rlike(r"^\d+$") &
#     (F.col("price_str").cast("int") >= 100) &
#     (F.col("price_str").cast("int") <= 150000),
#     F.col("price_str")
# ).otherwise(None)
# ).drop("price_str")

# # --- Odometer Cleaning ---
# # Handles: 120k | 120,000 | "120000 miles" | "high miles"
# df = df.withColumn("odo_str", F.lower(F.trim(F.col("odometer"))))
# df = df.withColumn("odo_str", F.regexp_replace(F.col("odo_str"), r"[,\s]", ""))
# df = df.withColumn("odo_str", F.regexp_replace(F.col("odo_str"), r"miles?", ""))
# df = df.withColumn(
#     "odo_str",
#     F.when(F.col("odo_str").rlike(r"^\d+k$"),
#            (F.regexp_replace(F.col("odo_str"), "k", "").cast("int") * 1000).cast("string"))
#     .otherwise(F.col("odo_str"))
# )
# df = df.withColumn("odo_str", F.regexp_extract(F.col("odo_str"), r"^(\d+)", 1))
# # df = df.withColumn(
# #     "odometer_clean",
#     # F.when(F.col("odo_str").rlike(r"^\d+$"), F.col("odo_str")).otherwise(None)
# df = df.withColumn(
#     "odometer_clean",
#     F.when(
#         F.col("odo_str").rlike(r"^\d+$") &
#         (F.col("odo_str").cast("int") >= 0) &
#         (F.col("odo_str").cast("int") <= 400000),
#         F.col("odo_str")
#     ).otherwise(None)
# ).drop("odo_str")
# # ).drop("odo_str")

# # --- Year Cleaning ---
# df = df.withColumn(
#     "year_clean",
#     F.when(F.col("year").rlike(r"^\d{4}$"), F.col("year")).otherwise(None)
# )

# log.info("  Price, odometer, year validated and cleaned.")

In [96]:
# =============================================================================
# 10. TYPE CONVERSION + CONVERSION FAILURE REPORT
# =============================================================================

In [97]:
log.info("Casting data types...")

price_total    = df.count()
year_total     = price_total
odo_total      = price_total

    # .withColumn("price",        F.col("price_clean").cast(IntegerType())) \
    # .withColumn("odometer",     F.col("odometer_clean").cast(IntegerType())) \
    # .withColumn("year",         F.col("year_clean").cast(IntegerType())) \
df = df \
    .withColumn("lat",          F.col("lat").cast(FloatType())) \
    .withColumn("long",         F.col("long").cast(FloatType())) \
    .withColumn("posting_date", F.to_timestamp(F.col("posting_date"))) \
    # .drop("price_clean", "odometer_clean", "year_clean")

price_fail    = df.filter(F.col("price").isNull()).count()
odometer_fail = df.filter(F.col("odometer").isNull()).count()
year_fail     = df.filter(F.col("year").isNull()).count()

log.info("  Type Conversion Report:")
log.info(f"    {'Column':<15} {'Total':>8} {'Failed':>8} {'Success%':>10}")
log.info(f"    {'price':<15} {price_total:>8} {price_fail:>8} {((price_total-price_fail)/price_total*100):>9.2f}%")
log.info(f"    {'odometer':<15} {odo_total:>8} {odometer_fail:>8} {((odo_total-odometer_fail)/odo_total*100):>9.2f}%")
log.info(f"    {'year':<15} {year_total:>8} {year_fail:>8} {((year_total-year_fail)/year_total*100):>9.2f}%")

2026-08-01 04:01:12 | INFO | Casting data types...
2026-08-01 04:01:18 | INFO |   Type Conversion Report:                          
2026-08-01 04:01:18 | INFO |     Column             Total   Failed   Success%
2026-08-01 04:01:18 | INFO |     price             425675        0    100.00%
2026-08-01 04:01:18 | INFO |     odometer          425675     4331     98.98%
2026-08-01 04:01:18 | INFO |     year              425675        0    100.00%


In [98]:
# Break query plan — accumulated transformations after type casting
df = df.checkpoint()
log.info("Checkpoint 2 written — query plan reset after type casting.")

2026-08-01 04:01:25 | INFO | Checkpoint 2 written — query plan reset after type casting.


In [99]:
# =============================================================================
# 11. ODOMETER MEDIAN IMPUTATION BY MANUFACTURER
#     (Outliers excluded before computing median)
# =============================================================================

In [100]:
log.info("Imputing missing odometer values using manufacturer group median...")

# Window specs
window_model = Window.partitionBy("manufacturer", "model")
window_manuf = Window.partitionBy("manufacturer")

# Compute layered medians
median_by_model = F.percentile_approx("odometer", 0.5).over(window_model)
median_by_manuf = F.percentile_approx("odometer", 0.5).over(window_manuf)

# Global median — computed as a scalar from non-null values
global_median_val = (
    df.filter(F.col("odometer").isNotNull())
      .select(F.percentile_approx("odometer", 0.5))
      .first()[0]
)

# Cascading coalesce: original → model median → manufacturer median → global
df = df.withColumn(
    "odometer",
    F.coalesce(
        F.col("odometer"),
        median_by_model,
        median_by_manuf,
        F.lit(global_median_val)
    )
)

log.info("  Odometer imputation complete (Model → Manufacturer → Global cascade).")

2026-08-01 04:01:25 | INFO | Imputing missing odometer values using manufacturer group median...
2026-08-01 04:01:28 | INFO |   Odometer imputation complete (Model → Manufacturer → Global cascade).


In [101]:
# =============================================================================
# 11b. TRANSMISSION MODE IMPUTATION BY MANUFACTURER + MODEL
# =============================================================================

log.info("Imputing missing transmission values using Manufacturer + Model mode...")

# Step 1: Treat "unknown" and empty as null so they participate in imputation
df = df.withColumn(
    "transmission_clean",
    F.when(
        F.col("transmission").isNull() | (F.col("transmission") == "unknown"),
        None
    ).otherwise(F.col("transmission"))
)

# Step 2: Count each valid transmission value per (manufacturer, model) group
trans_counts = df.filter(F.col("transmission_clean").isNotNull()) \
    .groupBy("manufacturer", "model", "transmission_clean") \
    .agg(F.count("*").alias("trans_count"))

# Step 3: Rank within each (manufacturer, model) group by frequency descending
#         transmission_clean as tie-breaker ensures determinism
window_rank = Window.partitionBy("manufacturer", "model") \
    .orderBy(F.col("trans_count").desc(), F.col("transmission_clean"))

trans_mode = trans_counts \
    .withColumn("rn", F.row_number().over(window_rank)) \
    .filter(F.col("rn") == 1) \
    .select(
        "manufacturer",
        "model",
        F.col("transmission_clean").alias("mode_transmission")
    )

# Step 4: Join the single definitive mode back to the main dataframe
df = df.join(trans_mode, on=["manufacturer", "model"], how="left")

# Step 5: Fill nulls with the mode; fallback to "automatic" if entire group had no data
df = df.withColumn(
    "transmission",
    F.coalesce(
        F.col("transmission_clean"),
        F.col("mode_transmission"),
        F.lit("automatic")
    )
).drop("transmission_clean", "mode_transmission")

log.info("  Transmission imputation complete (Manufacturer + Model mode → 'automatic' fallback).")

2026-08-01 04:01:28 | INFO | Imputing missing transmission values using Manufacturer + Model mode...
2026-08-01 04:01:28 | INFO |   Transmission imputation complete (Manufacturer + Model mode → 'automatic' fallback).


In [102]:
# =============================================================================
# 12. DUPLICATE REMOVAL
# =============================================================================

In [103]:
log.info("Removing duplicate records...")
count_before_dedup = df.count()

df = df.dropDuplicates(["id"])

df_with_vin    = df.filter(F.col("VIN").isNotNull())
df_without_vin = df.filter(F.col("VIN").isNull())
df_with_vin    = df_with_vin.dropDuplicates(["VIN"])
df             = df_with_vin.union(df_without_vin)

df = df.dropDuplicates(["url"])

count_after_dedup = df.count()
log.info(f"  Duplicates removed: {count_before_dedup - count_after_dedup}")

# Fuzzy Duplicate Report (composite key — report only, no hard drop)
fuzzy_dupes = df.groupBy("manufacturer", "model", "year", "price", "odometer") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count"))

fuzzy_dupe_groups = fuzzy_dupes.count()
log.info(f"  Fuzzy duplicate groups detected (same mfr+model+year+price+odo): {fuzzy_dupe_groups}")
log.info("  Note: Fuzzy duplicates reported only — not dropped (legitimate same-spec listings possible).")


2026-08-01 04:01:28 | INFO | Removing duplicate records...
2026-08-01 04:01:35 | INFO |   Duplicates removed: 147028                       
2026-08-01 04:01:43 | INFO |   Fuzzy duplicate groups detected (same mfr+model+year+price+odo): 16089
2026-08-01 04:01:43 | INFO |   Note: Fuzzy duplicates reported only — not dropped (legitimate same-spec listings possible).


In [104]:
# =============================================================================
# 13. BUSINESS RULE VALIDATION — STAGE 1
# =============================================================================

In [105]:

log.info("Applying business rule filters...")
count_before_rules = df.count()

# Price must exist and be positive
df = df.filter(F.col("price").isNotNull() & (F.col("price") > 0))

# Year range — dynamic upper bound
df = df.filter(
    F.col("year").isNotNull() &
    (F.col("year") >= 1980) &
    (F.col("year") <= CURRENT_YEAR)
)

# Coordinate validation — universal range first
# df = df.filter(
#     F.col("lat").isNull() | ((F.col("lat") >= -90.0) & (F.col("lat") <= 90.0))
# )
# df = df.filter(
#     F.col("long").isNull() | ((F.col("long") >= -180.0) & (F.col("long") <= 180.0))
# )

# US coordinate filter (dataset-specific — separated for reusability)
# US_LAT_MIN, US_LAT_MAX   = 24.0, 50.0
# US_LONG_MIN, US_LONG_MAX = -125.0, -66.0

# df = df.filter(
#     F.col("lat").isNull() |
#     ((F.col("lat") >= US_LAT_MIN) & (F.col("lat") <= US_LAT_MAX))
# )
# df = df.filter(
#     F.col("long").isNull() |
#     ((F.col("long") >= US_LONG_MIN) & (F.col("long") <= US_LONG_MAX))
# )

count_after_rules = df.count()
log.info(f"  Records removed by business rules: {count_before_rules - count_after_rules}")

2026-08-01 04:01:43 | INFO | Applying business rule filters...
2026-08-01 04:01:50 | INFO |   Records removed by business rules: 29603         


In [106]:
# =============================================================================
# 14. STATISTICAL OUTLIER REMOVAL — STAGE 2 (IQR-based)
# =============================================================================


In [107]:

# log.info("Applying IQR-based statistical outlier removal...")

# def remove_iqr_outliers(df, col_name, lower_pct=0.005, upper_pct=0.995):
#     bounds = df.filter(F.col(col_name).isNotNull()).approxQuantile(
#         col_name, [lower_pct, upper_pct], 0.01
#     )
#     lower, upper = bounds[0], bounds[1]
#     log.info(f"  IQR bounds [{col_name}]: lower={lower:.0f} | upper={upper:.0f}")
#     return df.filter(
#         F.col(col_name).isNull() |
#         ((F.col(col_name) >= lower) & (F.col(col_name) <= upper))
#     ), lower, upper

# count_before_iqr = df.count()
# df, price_lower, price_upper       = remove_iqr_outliers(df, "price")
# df, odometer_lower, odometer_upper = remove_iqr_outliers(df, "odometer")
# count_after_iqr = df.count()

# log.info(f"  Records removed by IQR outlier filter: {count_before_iqr - count_after_iqr}")



In [108]:
# =============================================================================
# 15. SAVE BRONZE VALIDATED (Intermediate Layer)
# =============================================================================


In [109]:

log.info(f"Writing Bronze Validated layer to: {BRONZE_VALIDATED}")
df.write.mode("overwrite").option("compression", "gzip").parquet(BRONZE_VALIDATED)
log.info("  Bronze Validated layer saved.")



2026-08-01 04:01:50 | INFO | Writing Bronze Validated layer to: hdfs:///user/vehicle_market/bronze_validated/vehicles_validated
2026-08-01 04:02:42 | INFO |   Bronze Validated layer saved.                    


In [110]:
# =============================================================================
# 16. WRITE SILVER LAYER
# =============================================================================

In [111]:
log.info(f"Writing Silver layer to: {SILVER_PATH}")
df.write.mode("overwrite").option("compression", "gzip").parquet(SILVER_PATH)
log.info("  Silver layer saved.")

2026-08-01 04:02:42 | INFO | Writing Silver layer to: hdfs:///user/vehicle_market/silver/vehicles_clean
2026-08-01 04:03:48 | INFO |   Silver layer saved.                              


In [112]:
df.count()

249044

In [113]:
# =============================================================================
# 17. SILVER LAYER VALIDATION REPORT
# =============================================================================

In [114]:

record_count_silver = df.count()
pipeline_end        = time.time()
execution_time      = round(pipeline_end - pipeline_start, 2)

log.info("\n" + "="*65)
log.info("           SILVER LAYER VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Bronze Records          : {record_count_bronze}")
log.info(f"  Silver Records          : {record_count_silver}")
log.info(f"  Total Records Removed   : {record_count_bronze - record_count_silver}")
log.info(f"  Columns Dropped         : {cols_dropped}")
log.info(f"  Columns Imputed         : ['manufacturer', 'model', 'odometer', 'fuel', 'transmission', 'title_status', 'state'] + Category B")
log.info(f"  Duplicates Removed      : {count_before_dedup - count_after_dedup}")
log.info(f"  Fuzzy Duplicate Groups  : {fuzzy_dupe_groups}")
log.info(f"  Business Rule Removed   : {count_before_rules - count_after_rules}")
# log.info(f"  IQR Outliers Removed    : {count_before_iqr - count_after_iqr}")
# log.info(f"  Price IQR Range         : {price_lower:.0f} — {price_upper:.0f}")
# log.info(f"  Odometer IQR Range      : {odometer_lower:.0f} — {odometer_upper:.0f}")
log.info(f"  Year Range Applied      : 1980 — {CURRENT_YEAR}")
log.info("")
log.info("  Type Conversion Failures:")
log.info(f"    price     : {price_fail}")
log.info(f"    odometer  : {odometer_fail}")
log.info(f"    year      : {year_fail}")
log.info("")
log.info("  Null Comparison (Before → After Cleaning):")
log.info(f"  {'Column':<20} {'Before':>10} {'After':>10}")
log.info(f"  {'-'*42}")

post_null_counts = {
    c: df.filter(F.col(c).isNull()).count()
    for c in df.columns
}

for col in df.columns:
    before = pre_null_counts.get(col, 0)
    after  = post_null_counts.get(col, 0)
    if before > 0 or after > 0:
        before_pct = before / record_count_bronze * 100
        after_pct  = after  / record_count_silver * 100
        log.info(f"  {col:<20} {before_pct:>9.2f}% {after_pct:>9.2f}%")

log.info("")
log.info("  Final Schema:")
df.printSchema()
log.info("="*65)
log.info("  Silver Layer Validation Complete.")
log.info("="*65)

2026-08-01 04:03:57 | INFO |                                                    
2026-08-01 04:03:57 | INFO |            SILVER LAYER VALIDATION REPORT
2026-08-01 04:03:57 | INFO | =================================================================
2026-08-01 04:03:57 | INFO |   Execution Time          : 318.6s
2026-08-01 04:03:57 | INFO |   Bronze Records          : 426880
2026-08-01 04:03:57 | INFO |   Silver Records          : 249044
2026-08-01 04:03:57 | INFO |   Total Records Removed   : 177836
2026-08-01 04:03:57 | INFO |   Columns Dropped         : ['county']
2026-08-01 04:03:57 | INFO |   Columns Imputed         : ['manufacturer', 'model', 'odometer', 'fuel', 'transmission', 'title_status', 'state'] + Category B
2026-08-01 04:03:57 | INFO |   Duplicates Removed      : 147028
2026-08-01 04:03:57 | INFO |   Fuzzy Duplicate Groups  : 16089
2026-08-01 04:03:57 | INFO |   Business Rule Removed   : 29603
2026-08-01 04:03:57 | INFO |   Year Range Applied      : 1980 — 2027
2026-08-01 04

Py4JJavaError: An error occurred while calling o2826.count.
: org.apache.spark.SparkException: Multiple failures in stage materialization.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.multiFailuresInStageMaterializationError(QueryExecutionErrors.scala:2075)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec.cleanUpAndThrowException(AdaptiveSparkPlanExec.scala:793)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec.$anonfun$getFinalPhysicalPlan$1(AdaptiveSparkPlanExec.scala:317)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:900)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec.getFinalPhysicalPlan(AdaptiveSparkPlanExec.scala:256)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec.withFinalPlanUpdate(AdaptiveSparkPlanExec.scala:401)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec.executeCollect(AdaptiveSparkPlanExec.scala:374)
	at org.apache.spark.sql.Dataset.$anonfun$count$1(Da

In [ ]:
df.show(5)

In [ ]:
# ====================================================
# DATA VALIDATION AFTER NEW CSV PARSER
# ====================================================

In [ ]:
from pyspark.sql import functions as F

# ===========================
# Dataset Summary
# ===========================
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Rows    : {df_raw.count()}")
print(f"Columns : {len(df_raw.columns)}")
print(df_raw.printSchema())

In [ ]:
categorical_cols = [
    "state",
    "transmission",
    "title_status",
    "fuel",
    "drive",
    "condition",
    "type"
]

for col in categorical_cols:
    print("\n" + "="*70)
    print(f"{col.upper()}")

    print("Distinct Values :", df_raw.select(col).distinct().count())

    df_raw.groupBy(col) \
          .count() \
          .orderBy(F.desc("count")) \
          .show(30, truncate=False)

In [ ]:
keywords = [
    "address",
    "finance",
    "country",
    "ford",
    "lincoln",
    "http",
    "www",
    "call",
    "phone"
]

for col in categorical_cols:
    print("\nChecking:", col)

    suspicious = df_raw.filter(
        F.lower(F.col(col)).rlike("|".join(keywords))
    )

    print("Suspicious Rows:", suspicious.count())

    suspicious.select(col).show(10, truncate=False)

In [ ]:
valid_states = [
    "al","ak","az","ar","ca","co","ct","de","fl","ga",
    "hi","id","il","in","ia","ks","ky","la","me","md",
    "ma","mi","mn","ms","mo","mt","ne","nv","nh","nj",
    "nm","ny","nc","nd","oh","ok","or","pa","ri","sc",
    "sd","tn","tx","ut","vt","va","wa","wv","wi","wy","dc"
]

invalid_states = df_raw.filter(
    (~F.lower(F.col("state")).isin(valid_states))
    & F.col("state").isNotNull()
)

print("Invalid State Rows :", invalid_states.count())

invalid_states.select("state") \
              .distinct() \
              .orderBy("state") \
              .show(100, truncate=False)

In [ ]:
valid_transmission = ["automatic", "manual", "other"]

invalid_trans = df_raw.filter(
    (~F.lower(F.col("transmission")).isin(valid_transmission))
    & F.col("transmission").isNotNull()
)

print("Invalid Transmission Rows :", invalid_trans.count())

invalid_trans.select("transmission") \
             .distinct() \
             .show(100, truncate=False)

In [ ]:
valid_title = [
    "clean",
    "rebuilt",
    "salvage",
    "lien",
    "missing",
    "parts only"
]

invalid_title = df_raw.filter(
    (~F.lower(F.col("title_status")).isin(valid_title))
    & F.col("title_status").isNotNull()
)

print("Invalid Title Status Rows :", invalid_title.count())

invalid_title.select("title_status") \
             .distinct() \
             .show(100, truncate=False)

In [ ]:
for col in categorical_cols:
    print("\nChecking long values in:", col)

    df_raw.filter(
        F.length(F.col(col)) > 30
    ).select(col).show(20, truncate=False)

In [ ]:
print("="*60)
print("DUPLICATE CHECK - ALL COLUMNS")
print("="*60)

total_rows = df_raw.count()
unique_rows = df_raw.dropDuplicates().count()

print(f"Total Rows           : {total_rows}")
print(f"Unique Rows          : {unique_rows}")
print(f"Duplicate Rows       : {total_rows - unique_rows}")

In [ ]:
from pyspark.sql import functions as F

print("="*60)
print("DUPLICATE IDs")
print("="*60)

duplicate_ids = (
    df_raw.groupBy("id")
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate ID Groups :", duplicate_ids.count())

duplicate_ids.orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
print("="*60)
print("DUPLICATE VINs")
print("="*60)

duplicate_vins = (
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate VIN Groups :", duplicate_vins.count())

duplicate_vins.orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
print("="*60)
print("DUPLICATE URLs")
print("="*60)

duplicate_urls = (
    df_raw.filter(F.col("url").isNotNull())
          .groupBy("url")
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate URL Groups :", duplicate_urls.count())

duplicate_urls.orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
print("="*60)
print("BEFORE & AFTER DUPLICATE REMOVAL")
print("="*60)

before = df_raw.count()

after = df_raw.dropDuplicates().count()     # or your duplicate logic

print(f"Rows Before : {before}")
print(f"Rows After  : {after}")
print(f"Removed     : {before - after}")

In [ ]:
print("="*70)
print("CHECKING DUPLICATES USING BUSINESS KEYS")
print("="*70)

business_keys = ["id", "VIN", "url"]

business_duplicates = (
    df_raw.groupBy(business_keys)
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate Business Records :", business_duplicates.count())

business_duplicates.orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
from pyspark.sql import functions as F

print("=" * 70)
print("VIN DUPLICATE ANALYSIS")
print("=" * 70)

# Total records
total_rows = df_raw.count()

# Rows with VIN
rows_with_vin = df_raw.filter(F.col("VIN").isNotNull()).count()

# Rows without VIN
rows_without_vin = df_raw.filter(F.col("VIN").isNull()).count()

# Unique VINs
unique_vins = (
    df_raw.filter(F.col("VIN").isNotNull())
          .select("VIN")
          .distinct()
          .count()
)

# Total rows having duplicate VINs
rows_in_duplicate_vins = (
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
          .agg(F.sum("count").alias("rows"))
          .collect()[0]["rows"]
)

# Number of VIN groups that are duplicated
duplicate_vin_groups = (
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
          .count()
)

# Expected rows removed if keeping one row per VIN
expected_removed = rows_with_vin - unique_vins

print(f"Total Rows                    : {total_rows:,}")
print(f"Rows with VIN                 : {rows_with_vin:,}")
print(f"Rows without VIN              : {rows_without_vin:,}")
print(f"Unique VINs                   : {unique_vins:,}")
print(f"Duplicate VIN Groups          : {duplicate_vin_groups:,}")
print(f"Rows belonging to Duplicate VINs : {rows_in_duplicate_vins:,}")
print(f"Expected Rows Removed         : {expected_removed:,}")

In [ ]:
from pyspark.sql import functions as F

(
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
          .orderBy(F.desc("count"))
          .show(20, truncate=False)
)